# A5 Table 1 — Baseline characteristics by outcome

This notebook creates an aggregate baseline-characteristics table from the locked extended feature matrix. The extended matrix contains the same frozen primary cohort and labels as the core matrix, while also retaining the temperature feature required for Table 1. It does not modify the frozen model and does not save patient-level data to Drive.


In [ ]:
from google.colab import auth
auth.authenticate_user()

from pathlib import Path
import os
import numpy as np
import pandas as pd
from google.cloud import bigquery

# Set these values for your own authorized BigQuery environment.
# No credentials or source data are included in this repository.
PROJECT_ID = os.environ.get("GOOGLE_CLOUD_PROJECT")
if not PROJECT_ID:
    PROJECT_ID = input("Enter your Google Cloud project ID: ").strip()

DATASET_ID = os.environ.get("AKI_DATASET_ID", "aki_jcmc_v2")
LOCATION = os.environ.get("BIGQUERY_LOCATION", "US")

if not PROJECT_ID:
    raise ValueError("A Google Cloud project ID is required.")

CORE_TABLE = f"{PROJECT_ID}.{DATASET_ID}.feature_matrix_core_outerfold_v1"

# Aggregate outputs are written to the notebook runtime by default.
# Users may replace this with an authorized persistent path.
OUT = Path(os.environ.get("AKI_OUTPUT_DIR", "/content/aki_table1_outputs"))
OUT.mkdir(parents=True, exist_ok=True)

client = bigquery.Client(project=PROJECT_ID)

print("Configured dataset:", f"{PROJECT_ID}.{DATASET_ID}")
print("Aggregate output directory:", OUT)

In [ ]:
# Table 1 requires temperature, which is available in the locked
# extended feature matrix rather than the reduced core modeling matrix.
# The cohort, labels, hospital groups, and 12-hour feature windows remain
# identical to the frozen primary cohort.

EXTENDED_TABLE = (
    f"{PROJECT_ID}.{DATASET_ID}."
    "feature_matrix_extended_outerfold_v1"
)

FEATURES = [
    "id_row",
    "label_stage23",
    "group_hospital",
    "x_age_years",
    "x_sex",
    "x_bmi",
    "x_reference_creatinine",
    "x_stage1_at_landmark",
    "x_lab_creatinine_last",
    "x_lab_bun_last",
    "x_vital_heart_rate_last",
    "x_vital_noninvasive_systolic_bp_last",
    "x_vital_respiratory_rate_last",
    "x_vital_temperature_c_last",
]

# Confirm that every requested Table 1 field is present.
table = client.get_table(EXTENDED_TABLE)
available_columns = {field.name for field in table.schema}
missing_columns = [name for name in FEATURES if name not in available_columns]

if missing_columns:
    raise KeyError(
        "The locked extended matrix is missing required Table 1 fields: "
        + ", ".join(missing_columns)
    )

sql = (
    "SELECT "
    + ", ".join(f"`{name}`" for name in FEATURES)
    + f" FROM `{EXTENDED_TABLE}`"
)

df = client.query(
    sql,
    location=LOCATION
).to_dataframe(create_bqstorage_client=True)

assert len(df) == 58491
assert df["id_row"].nunique() == 58491
assert int(df["label_stage23"].sum()) == 3032
assert df["group_hospital"].nunique() == 198

print("Table 1 source:", EXTENDED_TABLE)
print("Patients:", len(df))
print("Events:", int(df["label_stage23"].sum()))
print("Hospitals:", df["group_hospital"].nunique())
print("Locked cohort integrity: PASS")

In [ ]:
continuous = [
    ("Age, years", "x_age_years"),
    ("Body mass index, kg/m²", "x_bmi"),
    ("Reference creatinine, mg/dL", "x_reference_creatinine"),
    ("Last creatinine by 12 h, mg/dL", "x_lab_creatinine_last"),
    ("Last blood urea nitrogen by 12 h, mg/dL", "x_lab_bun_last"),
    ("Last heart rate by 12 h, beats/min", "x_vital_heart_rate_last"),
    ("Last systolic blood pressure by 12 h, mmHg", "x_vital_noninvasive_systolic_bp_last"),
    ("Last respiratory rate by 12 h, breaths/min", "x_vital_respiratory_rate_last"),
    ("Last temperature by 12 h, °C", "x_vital_temperature_c_last"),
]

def fmt_cont(s):
    s=s.dropna().astype(float)
    if len(s)==0: return "NA"
    return f"{s.median():.1f} ({s.quantile(.25):.1f}–{s.quantile(.75):.1f})"

def fmt_cat(mask, denom):
    n=int(mask.sum())
    return f"{n:,} ({100*n/denom:.1f}%)"

rows=[]
for label,col in continuous:
    rows.append({
        "Characteristic":label,
        "Overall (n=58,491)":fmt_cont(df[col]),
        "No progression (n=55,459)":fmt_cont(df.loc[df.label_stage23==0,col]),
        "Stage 2–3 progression (n=3,032)":fmt_cont(df.loc[df.label_stage23==1,col]),
        "Missing overall, n (%)":fmt_cat(df[col].isna(),len(df)),
    })

sex=df["x_sex"].fillna("other_or_missing").astype(str).str.lower()
for value,label in [("female","Female sex"),("male","Male sex"),("other_or_missing","Other/missing sex")]:
    rows.append({
        "Characteristic":label,
        "Overall (n=58,491)":fmt_cat(sex.eq(value),len(df)),
        "No progression (n=55,459)":fmt_cat(sex[df.label_stage23==0].eq(value),(df.label_stage23==0).sum()),
        "Stage 2–3 progression (n=3,032)":fmt_cat(sex[df.label_stage23==1].eq(value),(df.label_stage23==1).sum()),
        "Missing overall, n (%)":"—",
    })

stage1=df["x_stage1_at_landmark"].fillna(0).astype(float).eq(1)
rows.append({
    "Characteristic":"KDIGO stage 1 at the 12-h landmark",
    "Overall (n=58,491)":fmt_cat(stage1,len(df)),
    "No progression (n=55,459)":fmt_cat(stage1[df.label_stage23==0],(df.label_stage23==0).sum()),
    "Stage 2–3 progression (n=3,032)":fmt_cat(stage1[df.label_stage23==1],(df.label_stage23==1).sum()),
    "Missing overall, n (%)":fmt_cat(df["x_stage1_at_landmark"].isna(),len(df)),
})

table1=pd.DataFrame(rows)
display(table1)
table1.to_csv(OUT/"A5_Table1_baseline_characteristics_by_outcome.csv",index=False)
print("saved:",OUT/"A5_Table1_baseline_characteristics_by_outcome.csv")

In [ ]:
# Aggregate integrity summary.
summary=pd.DataFrame({
    "metric":["patients","events","hospitals","event_rate"],
    "value":[len(df),int(df.label_stage23.sum()),df.group_hospital.nunique(),df.label_stage23.mean()],
})
summary.to_csv(OUT/"A5_Table1_integrity_summary.csv",index=False)
print("A5 TABLE 1 RUN: COMPLETE")